# 09b - Auditoria da inferencia municipal no AIPW temporal

## Resumo executivo

O maior deslocamento individual LOCO de C1 sem trimming foi 0,034638 p.p. (2016).
Na exclusao conjunta dos dez municipios com maior U_g^2, o maior deslocamento foi
0,120544 p.p. (2018; 13,1% da populacao daquele ano removida). Nenhum efeito mudou
de sinal e nenhum IC municipal passou a incluir zero. O ponto estimado e numericamente
mais estavel que a precisao em anos com alta concentracao municipal.

Esta auditoria usa exclusivamente dados e previsoes OOF persistidos. As exclusoes
mantem os nuisances fixos; alteram a populacao observada e nao definem um novo
estimando causal principal. O limiar de top1_if2 permanece 0,50. Em 2024 a metrica
tambem excede esse limiar nas seis combinacoes, mas o fluxo de controle positivo nao
chama a rotina de sinalizacao aplicada a 2014-2023. A interpretacao diagnostica aqui
e simetrica para os 11 anos.

## 1. Contrato e fontes

- Unidade: registro de nascido vivo; tratamento T=1 para inicio do pre-natal ate o terceiro mes.
- C1 principal sem trimming: 11 anos, todos exibidos. C2/C3: somente 2016, com e sem trimming.
- LOCO: excluir um municipio por vez entre os dez maiores `U_g^2` do proprio ano/especificacao/recorte.
- Stress conjunto: excluir os cinco e dez maiores `U_g^2`; a populacao alvo muda.
- `U_g = soma(psi_i no municipio g) - n_g * media(psi)`; SE municipal e IC com o quantil t de G-1 graus de liberdade, conforme `src.robustez.resumo_cluster`.
- Fontes: `outputs/temporal/estimativas/aipw_anual.csv`, `ano_*.json`, `auditoria_influencia.csv`, caches OOF v2 e caches certificados de 2024. Nenhum ajuste ou selecao de nuisance e executado.

In [1]:
from pathlib import Path
from hashlib import sha256
import gc
import json
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import t as student_t
from IPython.display import display
from src.estimativas_temporais import (
    RAIZ, carregar_dados_ano, particoes, predicoes_certificadas_2024,
    hash_arquivo, hash_vetor,
)
from src.inferencia_causal import calcular_aipw
from src.robustez import resumo_cluster
from src.visualizacao import aplicar_tema_base

raiz = Path(RAIZ)
pasta = raiz / "outputs/temporal/estimativas"
cache = raiz / "data/interim/estimativas_temporais/v2"
aipw_original = pd.read_csv(pasta / "aipw_anual.csv")
assert len(aipw_original) == 66
arquivos_protegidos = [
    *(pasta / nome for nome in (
        "aipw_anual.csv", "aipw_janela_principal.csv",
        "aipw_sensibilidade_2018_2024.csv",
        "diagnosticos_nuisance_anual.csv", "inferencia_anual.csv",
        "execucao_anos.csv", "manifesto_notebook_09.json",
        "reconciliacao_2024.json")),
    *(pasta / f"ano_{ano}.json" for ano in range(2014, 2024)),
    raiz / "notebooks/09_estabilidade_temporal_estimativas.ipynb",
    raiz / "src/estimativas_temporais.py",
]
hashes_antes = {str(p.relative_to(raiz)): hash_arquivo(p) for p in arquivos_protegidos}
print(f"Fontes protegidas: {len(hashes_antes)} arquivos; AIPW anual: {len(aipw_original)} linhas.")

Fontes protegidas: 20 arquivos; AIPW anual: 66 linhas.


## 2. Leitura estrita dos checkpoints e formula de exclusao

Cada cache e conferido pelo hash do arquivo, hash da previsao, chave e indice de validacao. A previsao OOF so e posicionada no fold correspondente. O calculo usa `psi` ja definido pelo AIPW historico; excluir registros e tomar a media dos `psi` restantes e exatamente recalcular AIPW com os nuisances fixos.

In [2]:
def carregar_predicoes_existentes(ano, dados, entrada, nomes):
    if ano == 2024:
        previsoes, _ = predicoes_certificadas_2024(raiz, dados, entrada)
        return {nome: previsoes[nome] for nome in nomes}
    salvo = json.loads((pasta / f"ano_{ano}.json").read_text(encoding="utf-8"))
    for campo in ("n", "n_t0", "n_t1", "parquet_sha256",
                  "chave_sha256", "t_sha256", "y_sha256", "ordem"):
        assert entrada[campo] == salvo["entrada"][campo], (ano, campo)
    folds, auditoria = particoes(
        dados.tratamento.to_numpy(dtype=int),
        dados.Y_BAIXO_PESO.to_numpy(dtype=int),
    )
    assert auditoria == salvo["folds"], ano
    previsoes = {nome: np.full(len(dados), np.nan) for nome in nomes}
    for numero, (_, validacao) in enumerate(folds, 1):
        for nome in nomes:
            prefixo = cache / str(ano) / f"{nome}_fold{numero}"
            metadados = json.loads(prefixo.with_suffix(".json").read_text(encoding="utf-8"))
            assert metadados["status"] == "CONCLUIDO"
            assert metadados["contrato"]["chave_sha256"] == entrada["chave_sha256"]
            assert metadados["contrato"]["hash_validacao"] == auditoria["folds"][numero - 1]["hash_validacao"]
            assert hash_arquivo(prefixo.with_suffix(".npz")) == metadados["arquivo_sha256"]
            with np.load(prefixo.with_suffix(".npz")) as arquivo:
                valores = arquivo["pred"]
            assert len(valores) == len(validacao)
            assert hash_vetor(valores) == metadados["pred_sha256"]
            previsoes[nome][validacao] = valores
    assert all(np.isfinite(v).all() for v in previsoes.values()), ano
    return previsoes


def tabela_clusters(psi, municipios):
    nomes, codigos = np.unique(np.asarray(municipios, dtype=str), return_inverse=True)
    n_g = np.bincount(codigos)
    soma_g = np.bincount(codigos, weights=psi)
    theta = float(np.mean(psi))
    u_g = soma_g - n_g * theta
    u2_g = u_g ** 2
    tabela = pd.DataFrame({
        "municipio": nomes, "n_g": n_g, "soma_psi": soma_g,
        "share_n": n_g / len(psi), "U_g": u_g, "U_g2": u2_g,
        "share_variancia": u2_g / u2_g.sum(),
    })
    tabela["razao_concentracao"] = tabela.share_variancia / tabela.share_n
    return tabela.sort_values(["U_g2", "municipio"], ascending=[False, True]).reset_index(drop=True)


def inferencia_com_nuisances_fixos(tabela, municipios_excluidos=()):
    restante = tabela.loc[~tabela.municipio.isin(municipios_excluidos)]
    n = int(restante.n_g.sum())
    g = len(restante)
    assert n > 1 and g > 1
    theta = float(restante.soma_psi.sum() / n)
    u = restante.soma_psi.to_numpy() - restante.n_g.to_numpy() * theta
    se = float(np.sqrt(g / (g - 1) * np.dot(u, u) / n ** 2))
    critico = float(student_t.ppf(.975, g - 1))
    return {
        "n": n, "n_municipios": g, "aipw_pp": 100 * theta,
        "se_cluster_pp": 100 * se,
        "ic95_cluster_inferior_pp": 100 * (theta - critico * se),
        "ic95_cluster_superior_pp": 100 * (theta + critico * se),
    }


# Checagem independente em fixture sintetica, inclusive remocao de um cluster.
psi_teste = np.array([.03, -.01, .02, .08, -.04, .05], dtype=float)
grupos_teste = np.array(["A", "A", "B", "B", "C", "C"])
tab_teste = tabela_clusters(psi_teste, grupos_teste)
base_teste = inferencia_com_nuisances_fixos(tab_teste)
historico_teste = resumo_cluster(psi_teste, grupos_teste)
assert abs(base_teste["aipw_pp"] - historico_teste["estimativa_pp"]) < 1e-12
assert abs(base_teste["se_cluster_pp"] - historico_teste["se_cluster_pp"]) < 1e-12
menos_a = inferencia_com_nuisances_fixos(tab_teste, ["A"])
direto = resumo_cluster(psi_teste[grupos_teste != "A"], grupos_teste[grupos_teste != "A"])
assert abs(menos_a["aipw_pp"] - direto["estimativa_pp"]) < 1e-12
assert abs(menos_a["se_cluster_pp"] - direto["se_cluster_pp"]) < 1e-12
print("Fixture LOCO e SE municipal: OK")

Fixture LOCO e SE municipal: OK


## 3. C1 em todos os anos; confirmacao C2/C3 em 2016

As dez remocoes individuais e os stress tests sao definidos exclusivamente por `U_g^2` do resultado original de cada combinacao. Nao se seleciona municipio pela variacao LOCO observada.

In [3]:
linhas_tamanho = []
linhas_loco = []
linhas_stress = []

def auditar_combinacao(ano, especificacao, trimming, dados, previsoes):
    nome_e, nome_m = {
        "C1": ("e", "C1"), "C2": ("e", "C2"), "C3": ("e_C3", "C2")
    }[especificacao]
    e = previsoes[nome_e]
    m0, m1 = previsoes[f"m0_{nome_m}"], previsoes[f"m1_{nome_m}"]
    t = dados.tratamento.to_numpy(dtype=int)
    y = dados.Y_BAIXO_PESO.to_numpy(dtype=int)
    municipios = dados.CODMUNRES.to_numpy(dtype=str)
    lo, hi = (0., 1.) if trimming == "sem_trimming" else (.05, .95)
    mascara = (e >= lo) & (e <= hi)
    resultado, psi = calcular_aipw(y[mascara], t[mascara],
                                  e[mascara], m0[mascara], m1[mascara])
    tabela = tabela_clusters(psi, municipios[mascara])
    original = inferencia_com_nuisances_fixos(tabela)
    salvo = aipw_original.loc[
        aipw_original.ano.eq(ano)
        & aipw_original.especificacao.eq(especificacao)
        & aipw_original.trimming.eq(trimming)
    ].iloc[0]
    assert original["n"] == int(salvo.n) == resultado["n"]
    assert original["n_municipios"] == int(salvo.n_clusters)
    assert abs(original["aipw_pp"] - float(salvo.estimativa_pp)) < 1e-10
    assert abs(resultado["estimativa_pp"] - float(salvo.estimativa_pp)) < 1e-12
    for campo in ("se_cluster_pp", "ic95_cluster_inferior_pp",
                  "ic95_cluster_superior_pp"):
        assert abs(original[campo] - float(salvo[campo])) < 1e-10, (ano, especificacao, trimming, campo)
    top10 = tabela.head(10).copy()
    top10.insert(0, "rank_u2", range(1, 11))
    if especificacao == "C1" and trimming == "sem_trimming":
        for linha in top10.itertuples():
            linhas_tamanho.append({
                "ano": ano, "especificacao": especificacao,
                "trimming": trimming, "rank_u2": linha.rank_u2,
                "municipio": linha.municipio, "n_g": int(linha.n_g),
                "share_n": float(linha.share_n), "U_g": float(linha.U_g),
                "U_g2": float(linha.U_g2),
                "share_variancia": float(linha.share_variancia),
                "razao_concentracao": float(linha.razao_concentracao),
            })
    for linha in top10.itertuples():
        sem = inferencia_com_nuisances_fixos(tabela, [linha.municipio])
        # Verificacao direta do AIPW com os mesmos arrays OOF para o primeiro municipio.
        if linha.rank_u2 == 1:
            mantidos = municipios[mascara] != linha.municipio
            direto, _ = calcular_aipw(y[mascara][mantidos], t[mascara][mantidos],
                                     e[mascara][mantidos], m0[mascara][mantidos],
                                     m1[mascara][mantidos])
            assert abs(sem["aipw_pp"] - direto["estimativa_pp"]) < 1e-10
        linhas_loco.append({
            "ano": ano, "especificacao": especificacao, "trimming": trimming,
            "classe": "DIAGNOSTICO_DE_INFLUENCIA_COM_NUISANCES_FIXOS",
            "rank_u2": int(linha.rank_u2), "municipio": linha.municipio,
            "n_removido": int(linha.n_g), "share_n_removido": float(linha.share_n),
            "share_u2_municipio": float(linha.share_variancia),
            "aipw_original_pp": original["aipw_pp"],
            "aipw_sem_municipio_pp": sem["aipw_pp"],
            "delta_pp": sem["aipw_pp"] - original["aipw_pp"],
            "se_cluster_original_pp": original["se_cluster_pp"],
            "se_cluster_sem_municipio_pp": sem["se_cluster_pp"],
            "ic95_original_inferior_pp": original["ic95_cluster_inferior_pp"],
            "ic95_original_superior_pp": original["ic95_cluster_superior_pp"],
            "ic95_sem_municipio_inferior_pp": sem["ic95_cluster_inferior_pp"],
            "ic95_sem_municipio_superior_pp": sem["ic95_cluster_superior_pp"],
            "sinal_mudou": bool(np.sign(sem["aipw_pp"]) != np.sign(original["aipw_pp"])),
            "ic_passa_a_incluir_zero": bool(
                original["ic95_cluster_superior_pp"] < 0
                and sem["ic95_cluster_inferior_pp"] <= 0 <= sem["ic95_cluster_superior_pp"]
            ),
        })
    for k in (5, 10):
        excluidos = top10.municipio.iloc[:k].tolist()
        sem = inferencia_com_nuisances_fixos(tabela, excluidos)
        removido = int(top10.n_g.iloc[:k].sum())
        assert sem["n"] == original["n"] - removido
        linhas_stress.append({
            "ano": ano, "especificacao": especificacao, "trimming": trimming,
            "classe": "DIAGNOSTICO_DE_INFLUENCIA_COM_NUISANCES_FIXOS",
            "populacao_alvo_alterada": True, "top_k": k,
            "municipios_excluidos": "|".join(excluidos),
            "n_removido": removido, "share_n_removido": removido / original["n"],
            "share_u2_original_removida": float(top10.share_variancia.iloc[:k].sum()),
            "aipw_original_pp": original["aipw_pp"],
            "aipw_sem_clusters_pp": sem["aipw_pp"],
            "delta_pp": sem["aipw_pp"] - original["aipw_pp"],
            "se_cluster_original_pp": original["se_cluster_pp"],
            "se_cluster_sem_clusters_pp": sem["se_cluster_pp"],
            "ic95_original_inferior_pp": original["ic95_cluster_inferior_pp"],
            "ic95_original_superior_pp": original["ic95_cluster_superior_pp"],
            "ic95_sem_clusters_inferior_pp": sem["ic95_cluster_inferior_pp"],
            "ic95_sem_clusters_superior_pp": sem["ic95_cluster_superior_pp"],
            "sinal_mudou": bool(np.sign(sem["aipw_pp"]) != np.sign(original["aipw_pp"])),
            "ic_passa_a_incluir_zero": bool(
                original["ic95_cluster_superior_pp"] < 0
                and sem["ic95_cluster_inferior_pp"] <= 0 <= sem["ic95_cluster_superior_pp"]
            ),
        })

for ano in range(2014, 2025):
    dados, entrada = carregar_dados_ano(raiz, ano)
    nomes = ["e", "m0_C1", "m1_C1"]
    if ano == 2016:
        nomes += ["e_C3", "m0_C2", "m1_C2"]
    previsoes = carregar_predicoes_existentes(ano, dados, entrada, nomes)
    auditar_combinacao(ano, "C1", "sem_trimming", dados, previsoes)
    if ano == 2016:
        for especificacao in ("C2", "C3"):
            for trimming in ("sem_trimming", "0.05_0.95"):
                auditar_combinacao(ano, especificacao, trimming, dados, previsoes)
    print(f"{ano}: C1 LOCO Top 10 e stress Top 5/10 reconciliados; nenhum refit.", flush=True)
    del dados, previsoes
    gc.collect()

tamanho = pd.DataFrame(linhas_tamanho)
loco = pd.DataFrame(linhas_loco)
stress = pd.DataFrame(linhas_stress)
assert len(tamanho) == 110 and len(loco) == 150 and len(stress) == 30
assert set(tamanho.ano) == set(range(2014, 2025))
for nome, tabela in (
    ("auditoria_loco_municipios.csv", loco),
    ("auditoria_stress_clusters.csv", stress),
    ("auditoria_cluster_tamanho.csv", tamanho),
):
    tabela.to_csv(pasta / nome, index=False, float_format="%.17g")
print(f"Top clusters: {len(tamanho)}; LOCO: {len(loco)}; stress: {len(stress)}; checkpoints agregados salvos.")

2014 | preparo | N=2691119 | 23.9s | RSS=1713238016


2014: C1 LOCO Top 10 e stress Top 5/10 reconciliados; nenhum refit.


2015 | preparo | N=2745846 | 25.3s | RSS=2097823744


2015: C1 LOCO Top 10 e stress Top 5/10 reconciliados; nenhum refit.


2016 | preparo | N=2626594 | 25.1s | RSS=2097823744


2016: C1 LOCO Top 10 e stress Top 5/10 reconciliados; nenhum refit.


... (13 outputs left out)

2023: C1 LOCO Top 10 e stress Top 5/10 reconciliados; nenhum refit.


2024 | preparo | N=2251570 | 26.6s | RSS=2347450368


2024: C1 LOCO Top 10 e stress Top 5/10 reconciliados; nenhum refit.


Top clusters: 110; LOCO: 150; stress: 30; checkpoints agregados salvos.


## 4. Resumo anual e persistencia

`max |delta|` e a mediana descrevem as remocoes observadas, sem impor novo limiar. O SE e o IC sao recalculados na populacao remanescente. A variacao do efeito e a variacao da precisao sao lidas separadamente.

In [4]:
resumo_linhas = []
for (ano, spec, trimming), grupo in loco.groupby(["ano", "especificacao", "trimming"], sort=True):
    idx = grupo.delta_pp.abs().idxmax()
    maior = grupo.loc[idx]
    resumo_linhas.append({
        "ano": int(ano), "especificacao": spec, "trimming": trimming,
        "max_delta_abs_pp": float(grupo.delta_pp.abs().max()),
        "mediana_delta_pp": float(grupo.delta_pp.median()),
        "mediana_delta_abs_pp": float(grupo.delta_pp.abs().median()),
        "municipio_max_delta": str(maior.municipio),
        "max_abs_variacao_se_pp": float(
            (grupo.se_cluster_sem_municipio_pp - grupo.se_cluster_original_pp).abs().max()),
        "algum_sinal_mudou": bool(grupo.sinal_mudou.any()),
        "algum_ic_passa_a_incluir_zero": bool(grupo.ic_passa_a_incluir_zero.any()),
        "aipw_original_pp": float(maior.aipw_original_pp),
        "se_cluster_original_pp": float(maior.se_cluster_original_pp),
    })
resumo = pd.DataFrame(resumo_linhas)
assert len(resumo) == 15
for nome, tabela in (
    ("auditoria_loco_municipios.csv", loco),
    ("auditoria_loco_resumo.csv", resumo),
    ("auditoria_stress_clusters.csv", stress),
    ("auditoria_cluster_tamanho.csv", tamanho),
):
    tabela.to_csv(pasta / nome, index=False, float_format="%.17g")
proveniencia = {
    "classe": "DIAGNOSTICO_DE_INFLUENCIA_COM_NUISANCES_FIXOS",
    "anos_c1": list(range(2014, 2025)),
    "c2_c3_apenas_2016": ["sem_trimming", "0.05_0.95"],
    "loco_criterio": "10 maiores U_g^2 por ano/especificacao/trimming",
    "stress_criterio": "5 e 10 maiores U_g^2",
    "populacao_alvo_stress_alterada": True,
    "n_linhas": {"tamanho": len(tamanho), "loco": len(loco),
                 "resumo": len(resumo), "stress": len(stress)},
    "resultado_2024": "top1_if2 excede 0.50 nas seis combinacoes, apesar de status CONCLUIDO do controle positivo; rotina sinalizacoes nao e chamada para 2024",
    "nuisances": "predicoes OOF existentes; zero ajustes",
    "hashes_protegidos_antes": hashes_antes,
}
hashes_depois = {str(p.relative_to(raiz)): hash_arquivo(p) for p in arquivos_protegidos}
assert hashes_antes == hashes_depois
proveniencia["hashes_protegidos_depois"] = hashes_depois
proveniencia["hashes_preservados"] = True
(pasta / "auditoria_inferencia_09b.json").write_text(
    json.dumps(proveniencia, ensure_ascii=False, indent=2) + chr(10),
    encoding="utf-8",
)
display(resumo.loc[
    resumo.especificacao.eq("C1"),
    ["ano", "max_delta_abs_pp", "mediana_delta_pp", "municipio_max_delta",
     "max_abs_variacao_se_pp", "algum_sinal_mudou", "algum_ic_passa_a_incluir_zero"]
].sort_values("ano").reset_index(drop=True))

,ano,max_delta_abs_pp,mediana_delta_pp,municipio_max_delta,max_abs_variacao_se_pp,algum_sinal_mudou,algum_ic_passa_a_incluir_zero
0,2014,0.016343,0.007122,355030,0.001220,False,False
1,2015,0.017071,0.007125,330455,0.002742,False,False
2,2016,0.034638,0.008222,330455,0.005936,False,False
3,2017,0.028109,0.008070,350950,0.004999,False,False
4,2018,0.029066,0.012638,355030,0.002188,False,False
5,2019,0.018361,0.008396,431490,0.003095,False,False
6,2020,0.032543,0.007466,330455,0.006526,False,False
7,2021,0.033712,0.007093,355030,0.004534,False,False
8,2022,0.022558,0.008666,355030,0.001514,False,False
9,2023,0.017003,0.007357,355030,0.001738,False,False


## 5. Efeito pontual versus inferencia

A primeira figura mostra o maior deslocamento LOCO do ponto estimado em cada ano. A segunda confronta participacao na populacao e na variancia municipal para os dez clusters com maior `U_g^2`. Uma posicao acima da diagonal indica contribuicao desproporcional e variancia, sem implicar erro ou causalidade.

In [5]:
principal = resumo.loc[
    resumo.especificacao.eq("C1") & resumo.trimming.eq("sem_trimming")
].sort_values("ano")
fig_delta = go.Figure(go.Bar(
    x=principal.ano, y=principal.max_delta_abs_pp,
    marker_color="#00598E",
    customdata=np.array(list(zip(principal.municipio_max_delta, principal.max_abs_variacao_se_pp)), dtype=object),
    hovertemplate="Ano %{x}<br>Maior |delta| %{y:.6f} p.p.<br>Municipio %{customdata[0]}<br>Maior mudanca |SE| %{customdata[1]:.6f} p.p.<extra></extra>",
))
fig_delta.update_layout(
    xaxis_title="Ano", yaxis_title="Maior |delta LOCO| (p.p.)",
    xaxis={"tickmode": "linear", "dtick": 1},
    height=440, width=900,
)
aplicar_tema_base(fig_delta, "Maior alteracao LOCO do AIPW C1 por ano")
fig_delta.show()

In [6]:
fig_tamanho = go.Figure()
for ano, grupo in tamanho.groupby("ano", sort=True):
    fig_tamanho.add_trace(go.Scatter(
        x=grupo.share_n, y=grupo.share_variancia,
        mode="markers", name=str(ano),
        marker={"size": 9},
        customdata=np.array(list(zip(grupo.municipio, grupo.razao_concentracao, grupo.rank_u2)), dtype=object),
        hovertemplate="Ano " + str(ano) + "<br>Municipio %{customdata[0]}<br>Rank U^2 %{customdata[2]}<br>Share N %{x:.3%}<br>Share variancia %{y:.3%}<br>Razao %{customdata[1]:.2f}<extra></extra>",
    ))
limite = float(max(tamanho.share_n.max(), tamanho.share_variancia.max()) * 1.05)
fig_tamanho.add_trace(go.Scatter(
    x=[0, limite], y=[0, limite], mode="lines", name="Proporcao 1:1",
    line={"color": "#444444", "dash": "dot"}, hoverinfo="skip",
))
fig_tamanho.update_layout(
    xaxis_title="Participacao nos nascimentos do ano",
    yaxis_title="Participacao em soma municipal de U_g^2",
    xaxis={"range": [0, limite], "tickformat": ".0%"},
    yaxis={"range": [0, limite], "tickformat": ".0%"},
    height=580, width=900,
)
aplicar_tema_base(fig_tamanho, "Tamanho versus contribuicao dos dez municipios com maior U^2")
fig_tamanho.show()

## 6. Stress conjunto e anos prioritarios

Excluir cinco ou dez municipios altera a populacao observada. Os resultados abaixo servem apenas para medir dependencia do ponto e da precisao a esses clusters, com nuisance fixo. Incluem todos os 11 anos; 2016-2018 e 2020-2021 recebem atencao descritiva pela concentracao municipal pre-identificada.

In [7]:
display(stress.loc[
    stress.especificacao.eq("C1"),
    ["ano", "top_k", "n_removido", "share_n_removido",
     "share_u2_original_removida", "aipw_original_pp",
     "aipw_sem_clusters_pp", "delta_pp",
     "se_cluster_original_pp", "se_cluster_sem_clusters_pp",
     "ic95_sem_clusters_inferior_pp", "ic95_sem_clusters_superior_pp"]
].sort_values(["ano", "top_k"]).reset_index(drop=True))

,ano,top_k,n_removido,share_n_removido,share_u2_original_removida,aipw_original_pp,aipw_sem_clusters_pp,delta_pp,se_cluster_original_pp,se_cluster_sem_clusters_pp,ic95_sem_clusters_inferior_pp,ic95_sem_clusters_superior_pp
0,2014,5,252616,0.093870,0.218103,-0.849446,-0.825126,0.024320,0.053872,0.052568,-0.928180,-0.722072
1,2014,10,286938,0.106624,0.307965,-0.849446,-0.816599,0.032847,0.053872,0.050144,-0.914902,-0.718297
2,2015,5,311463,0.113431,0.254150,-0.881686,-0.838404,0.043282,0.052184,0.050825,-0.938041,-0.738767
3,2015,10,379292,0.138133,0.343153,-0.881686,-0.829405,0.052281,0.052184,0.049140,-0.925738,-0.733072
4,2016,5,163478,0.062240,0.441305,-0.982854,-0.940425,0.042429,0.072810,0.058672,-1.055446,-0.825405
5,2016,10,413439,0.157405,0.533031,-0.982854,-0.921944,0.060910,0.072810,0.059110,-1.037823,-0.806065
6,2017,5,296083,0.109836,0.376257,-1.003279,-0.933955,0.069324,0.066539,0.059531,-1.050659,-0.817251
7,2017,10,354408,0.131473,0.438814,-1.003279,-0.889567,0.113712,0.066539,0.057912,-1.003098,-0.776036
8,2018,5,272228,0.099554,0.403338,-1.120190,-1.007750,0.112440,0.073089,0.062500,-1.130274,-0.885226
9,2018,10,358900,0.131250,0.521859,-1.120190,-0.999646,0.120544,0.073089,0.058400,-1.114132,-0.885160


## 7. Confirmacao limitada C2/C3 em 2016; leitura 2021-2022

C2/C3 sao conferidas so em 2016, nos dois recortes pre-fixados. A diferenca 2021-2022 e descritiva: nao se faz teste formal de quebra nem atribuicao a evento externo.

In [8]:
display(resumo.loc[
    resumo.ano.eq(2016) & resumo.especificacao.isin(["C2", "C3"]),
    ["especificacao", "trimming", "max_delta_abs_pp",
     "municipio_max_delta", "max_abs_variacao_se_pp"]
].reset_index(drop=True))
display(stress.loc[
    stress.ano.eq(2016) & stress.especificacao.isin(["C2", "C3"]),
    ["especificacao", "trimming", "top_k", "share_n_removido",
     "delta_pp", "se_cluster_original_pp", "se_cluster_sem_clusters_pp"]
].reset_index(drop=True))
display(stress.loc[
    stress.ano.isin([2021, 2022]) & stress.especificacao.eq("C1"),
    ["ano", "top_k", "aipw_original_pp", "aipw_sem_clusters_pp",
     "delta_pp", "se_cluster_original_pp", "se_cluster_sem_clusters_pp"]
].sort_values(["ano", "top_k"]).reset_index(drop=True))

,especificacao,trimming,max_delta_abs_pp,municipio_max_delta,max_abs_variacao_se_pp
0,C2,0.05_0.95,0.039065,330455,0.008852
1,C2,sem_trimming,0.037537,330455,0.007077
2,C3,0.05_0.95,0.036489,330455,0.007516
3,C3,sem_trimming,0.038386,330455,0.007613


,especificacao,trimming,top_k,share_n_removido,delta_pp,se_cluster_original_pp,se_cluster_sem_clusters_pp
0,C2,sem_trimming,5,0.109449,0.082829,0.074645,0.060699
1,C2,sem_trimming,10,0.157013,0.067444,0.074645,0.058832
2,C2,0.05_0.95,5,0.062265,0.043188,0.070986,0.055149
3,C2,0.05_0.95,10,0.144145,0.061536,0.070986,0.054680
4,C3,sem_trimming,5,0.109449,0.084351,0.074156,0.059993
5,C3,sem_trimming,10,0.151236,0.067937,0.074156,0.057085
6,C3,0.05_0.95,5,0.060717,0.038592,0.070431,0.055678
7,C3,0.05_0.95,10,0.144699,0.060353,0.070431,0.054514


,ano,top_k,aipw_original_pp,aipw_sem_clusters_pp,delta_pp,se_cluster_original_pp,se_cluster_sem_clusters_pp
0,2021,5,-1.108588,-1.046628,0.061959,0.067727,0.059689
1,2021,10,-1.108588,-1.052671,0.055917,0.067727,0.058786
2,2022,5,-1.388228,-1.361115,0.027113,0.069809,0.065881
3,2022,10,-1.388228,-1.332239,0.055989,0.069809,0.063571


## 8. Interpretacao e limites

**Efeito pontual:** comparar deslocamentos LOCO e do stress em pontos percentuais; nao ha reestimacao dos nuisances. **Inferencia:** comparar o SE e o IC municipal recalculados apos exclusao. O diagnostico pode revelar precisao dependente de poucos municipios mesmo quando o efeito pontual muda pouco.

A retirada muda o conjunto de nascidos vivos e, no stress, a populacao alvo. LOCO nao prova robustez causal nem corrige confundimento nao medido. A concentracao municipal nao invalida automaticamente o AIPW. Nenhum limiar adicional e definido nesta rodada. Os arquivos AIPW originais e o notebook 09 permanecem intactos.